In [1]:
!pip install -q lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 22.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    MinMaxScaler,
    OrdinalEncoder,
    LabelEncoder
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, metrics

import lime
import lime.lime_tabular

**MENYIAPKAN DATASET**

In [3]:
# =========================
# [MOUNT GOOGLE DRIVE]
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# [SEED]
# =========================
#SEED = 42
#random.seed(SEED)
#np.random.seed(SEED)
#tf.random.set_seed(SEED)
#os.environ["PYTHONHASHSEED"] = str(SEED)

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

tf.config.experimental.enable_op_determinism()

# =========================
# [SET PROJECT ROOT - FIX COLAB]
# =========================
PROJECT_ROOT = Path("/content/drive/MyDrive/SEMHAS2")

TRANSACTION_PATH = PROJECT_ROOT / "train_transaction.csv"
IDENTITY_PATH    = PROJECT_ROOT / "train_identity.csv"

OUTPUT_ROOT = PROJECT_ROOT / "output"
OUTPUT_ROOT.mkdir(exist_ok=True)

MERGED_OUTPATH  = OUTPUT_ROOT / "train_merged_by_identity.csv"
METRICS_OUTPATH = OUTPUT_ROOT / "validation_metrics_full_pipeline.json"

QUICK_RUN = False
QUICK_RUN_SAMPLE = 100_000

# =========================
# [VALIDASI PATH]
# =========================
print("Transaction exists:", TRANSACTION_PATH.exists())
print("Identity exists:", IDENTITY_PATH.exists())
print("Current dir:", PROJECT_ROOT)

print("Memeriksa file transaksi dan identity...\n")

assert TRANSACTION_PATH.exists(), (
    f"[ERROR] File transaksi tidak ditemukan: {TRANSACTION_PATH}"
)
assert IDENTITY_PATH.exists(), (
    f"[ERROR] File identity tidak ditemukan: {IDENTITY_PATH}"
)

transaksi = pd.read_csv(TRANSACTION_PATH)
identity  = pd.read_csv(IDENTITY_PATH)

print(f"Transaction shape : {transaksi.shape}")
print(f"Identity    shape : {identity.shape}\n")

assert 'TransactionID' in transaksi.columns, (
    "[ERROR] Kolom TransactionID tidak ditemukan di train_transaction"
)
assert 'TransactionID' in identity.columns, (
    "[ERROR] Kolom TransactionID tidak ditemukan di train_identity"
)
assert 'isFraud' in transaksi.columns, (
    "[ERROR] Kolom isFraud tidak ditemukan di train_transaction"
)

Mounted at /content/drive
Transaction exists: True
Identity exists: True
Current dir: /content/drive/MyDrive/SEMHAS2
Memeriksa file transaksi dan identity...

Transaction shape : (590540, 394)
Identity    shape : (144233, 41)



**MERGRE DATASET**

In [4]:
print("Memeriksa apakah semua TransactionID di identity terdapat di transaction...\n")

iden_ids = identity['TransactionID']
txn_ids  = transaksi['TransactionID']

n_iden_rows     = len(identity)
n_iden_covered  = iden_ids.isin(txn_ids).sum()
cover_rate_iden = n_iden_covered / n_iden_rows

print(
    f"Identity rows               : {n_iden_rows:,}\n"
    f"Tercover di transaction     : {n_iden_covered:,}\n"
    f"Coverage rate               : {cover_rate_iden:.4%}\n"
)

if cover_rate_iden == 1.0:
    print("✔ Semua TransactionID di identity terdapat di transaction (100%).")
else:
    missing_from_txn = identity.loc[
        ~iden_ids.isin(txn_ids), 'TransactionID'
    ].unique()

    print("✖ Tidak semua TransactionID di identity ada di transaction.")
    print(f"  Jumlah identity yang tidak ter-cover: {len(missing_from_txn):,}")
    print(f"  Contoh 10 TransactionID yang hilang: {list(missing_from_txn[:10])}")


print("Memeriksa duplikat TransactionID pada identity...\n")

dup_counts = (
    identity
    .groupby('TransactionID')
    .size()
    .sort_values(ascending=False)
)

n_dup = (dup_counts > 1).sum()

if n_dup == 0:
    print("✔ Tidak ada duplikat TransactionID di identity.")
    identity_proc = identity.copy()
else:
    print(
        f"✖ Ditemukan {n_dup:,} TransactionID duplikat di identity.\n"
        "  Mengambil first occurrence (drop_duplicates keep='first').\n"
    )

    print("Contoh top duplikat:")
    print(dup_counts.head().to_string(), "\n")

    identity_proc = (
        identity
        .drop_duplicates(subset=['TransactionID'], keep='first')
        .reset_index(drop=True)
    )

    print(f"Shape identity setelah deduplikasi: {identity_proc.shape}")

Memeriksa apakah semua TransactionID di identity terdapat di transaction...

Identity rows               : 144,233
Tercover di transaction     : 144,233
Coverage rate               : 100.0000%

✔ Semua TransactionID di identity terdapat di transaction (100%).
Memeriksa duplikat TransactionID pada identity...

✔ Tidak ada duplikat TransactionID di identity.


In [5]:
print("Melakukan merge identity dan transaction...\n")

# =========================
# [MERGE STRATEGY]
# =========================
if cover_rate_iden == 1.0:
    merge_how = "left"
else:
    merge_how = "inner"

merged = identity_proc.merge(
    transaksi,
    on="TransactionID",
    how=merge_how,
    indicator=True
)

print(f"Hasil merge               : {merged.shape}")
print("Distribusi indikator merge:")
print(merged["_merge"].value_counts(), "\n")

# =========================
# [VALIDASI LABEL]
# =========================
assert "isFraud" in merged.columns, (
    "[ERROR] Kolom isFraud hilang setelah merge. "
    "Pastikan train_transaction masih memuat label sebelum merge."
)

# =========================
# [AMBIL LABEL]
# =========================
y_merged = merged["isFraud"].astype(int).copy()

n_total   = len(y_merged)
n_fraud   = int(y_merged.sum())
pct_fraud = (n_fraud / n_total) * 100.0 if n_total > 0 else 0.0

print(
    "Distribusi label hasil merge:\n"
    f"  Total data   : {n_total:,}\n"
    f"  Fraud        : {n_fraud:,}\n"
    f"  Fraud rate   : {pct_fraud:.6f}%\n"
)

# =========================
# [DROP LABEL DARI FITUR]
# =========================
merged = merged.drop(columns=["isFraud"])

print("Kolom fitur setelah merge:")
print(merged.columns.tolist())
print("\nLabel y_merged siap digunakan.")

print("Pemeriksaan QUICK_RUN (stratified subsample)...\n")

if QUICK_RUN:
    if y_merged is None:
        raise ValueError(
            "QUICK_RUN=True tetapi label isFraud tidak ditemukan; "
            "tidak dapat melakukan stratified sampling."
        )

    merged_tmp = merged.copy()
    merged_tmp["_label"] = y_merged.values

    total = len(merged_tmp)
    sample_n = min(QUICK_RUN_SAMPLE, total)

    frac_pos = merged_tmp["_label"].mean()
    n_pos = max(1, int(round(sample_n * frac_pos)))
    n_neg = max(1, sample_n - n_pos)

    df_pos = merged_tmp[merged_tmp["_label"] == 1]
    df_neg = merged_tmp[merged_tmp["_label"] == 0]

    if len(df_pos) == 0:
        pilih = merged_tmp.sample(
            n=sample_n,
            random_state=SEED
        )
    else:
        s_pos = df_pos.sample(
            n=min(n_pos, len(df_pos)),
            random_state=SEED
        )
        s_neg = df_neg.sample(
            n=min(n_neg, len(df_neg)),
            random_state=SEED
        )

        pilih = (
            pd.concat([s_pos, s_neg])
            .sample(frac=1, random_state=SEED)
        )

    # =========================
    # [REASSIGN FEATURE & LABEL]
    # =========================
    pilih = pilih.reset_index(drop=True)
    y_merged = pilih["_label"].astype(int).reset_index(drop=True)
    merged = pilih.drop(columns=["_label"]).reset_index(drop=True)

    print(
        f"QUICK_RUN aktif → subsample {len(merged):,} baris | "
        f"fraud_rate = {y_merged.mean():.6%}\n"
    )
else:
    print("QUICK_RUN tidak aktif. Menggunakan seluruh dataset.\n")

# =========================
# [LABEL DISTRIBUTION]
# =========================
label_counts = y_merged.value_counts().sort_index()
label_pct = y_merged.value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    "jumlah": label_counts,
    "persentase (%)": label_pct.round(2)
})

summary.index = ["Non-Fraud (0)", "Fraud (1)"]

print("Ringkasan distribusi label:")
print(summary)

Melakukan merge identity dan transaction...

Hasil merge               : (144233, 435)
Distribusi indikator merge:
_merge
both          144233
left_only          0
right_only         0
Name: count, dtype: int64 

Distribusi label hasil merge:
  Total data   : 144,233
  Fraud        : 11,318
  Fraud rate   : 7.847025%

Kolom fitur setelah merge:
['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1'

**PREPROCESSING DATA**

In [6]:
print("=== PREPROCESSING ===\n")

# =========================
# [COPY DATA]
# =========================
df = merged.copy()

# =========================
# [0] DROP KOLUMN MERGE (WAJIB)
# =========================
if "_merge" in df.columns:
    df.drop(columns=["_merge"], inplace=True)

# =========================
# [1] Drop Kolom dengan Missing Value Tinggi
# =========================
MISS_THRESH = 0.95
miss_ratio = df.isnull().mean()
cols_high_missing = miss_ratio[miss_ratio > MISS_THRESH].index.tolist()

print(f"[1] Kolom dengan missing > {MISS_THRESH*100:.0f}% : {len(cols_high_missing)}")

# =========================
# [2] Near-Constant Feature Removal (Kategori + Numerik)
# =========================
CONST_THRESH = 0.999

# Kategori
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

if len(cat_cols) > 0:
    dominant_frac_cat = df[cat_cols].apply(
        lambda x: x.value_counts(normalize=True, dropna=False).max()
    )
    cols_near_const_cat = dominant_frac_cat[
        dominant_frac_cat > CONST_THRESH
    ].index.tolist()
else:
    cols_near_const_cat = []

# Numerik
num_cols = df.select_dtypes(include=np.number).columns.tolist()

if len(num_cols) > 0:
    dominant_frac_num = df[num_cols].apply(
        lambda x: x.value_counts(normalize=True, dropna=False).max()
    )
    cols_near_const_num = dominant_frac_num[
        dominant_frac_num > CONST_THRESH
    ].index.tolist()
else:
    cols_near_const_num = []

cols_near_const = list(set(cols_near_const_cat + cols_near_const_num))

print(f"[2] Kolom near-constant (> {CONST_THRESH*100:.2f}%) : {len(cols_near_const)}")

# =========================
# [3] Drop Kolom Duplikat Hasil Merge
# =========================
dup_cols = [c for c in df.columns if c.endswith("_y")]

print(f"[3] Kolom duplikat hasil merge (_y) : {len(dup_cols)}")

# =========================
# [4] Gabungkan Kolom yang Akan Di-drop
# =========================
IMPORTANT_COLS = [
    "TransactionID", "TransactionDT", "TransactionAmt",
    "ProductCD", "card1", "card2", "card3", "card4"
]

to_drop = list(set(cols_high_missing + cols_near_const + dup_cols))
to_drop = [c for c in to_drop if c not in IMPORTANT_COLS]

print(f"[4] Total kolom yang di-drop : {len(to_drop)}")

df.drop(columns=to_drop, inplace=True)

print("Shape setelah feature selection:", df.shape)

# =========================
# [5] Hapus Duplikat Baris
# =========================
before_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_rows = len(df)

print(f"[5] Duplikat baris dihapus : {before_rows - after_rows}")
print("Shape setelah hapus duplikat baris:", df.shape)

# =========================
# [6] Feature Engineering Waktu
# =========================
if "TransactionDT" in df.columns:
    df["hour"] = ((df["TransactionDT"] // 3600) % 24).astype("int8")
    df["day"] = ((df["TransactionDT"] // (3600 * 24)) % 7).astype("int8")
    df["day_elapsed"] = (df["TransactionDT"] // (3600 * 24)).astype("int32")

    print("[6] Fitur waktu dibuat: hour, day, day_elapsed")
else:
    print("[6] TransactionDT tidak ditemukan — fitur waktu dilewati")

print("\nPreprocessing selesai.")
print("Final shape:", df.shape)

=== PREPROCESSING ===

[1] Kolom dengan missing > 95% : 30
[2] Kolom near-constant (> 99.90%) : 25
[3] Kolom duplikat hasil merge (_y) : 0
[4] Total kolom yang di-drop : 34
Shape setelah feature selection: (144233, 399)
[5] Duplikat baris dihapus : 0
Shape setelah hapus duplikat baris: (144233, 399)
[6] Fitur waktu dibuat: hour, day, day_elapsed

Preprocessing selesai.
Final shape: (144233, 402)


In [8]:
print("=== IMPUTATION: NUMERIK & KATEGORIKAL ===\n")

# =========================
# Identifikasi Tipe Fitur
# =========================
num_cols = df.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

cat_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(f"Jumlah fitur numerik     : {len(num_cols)}")
print(f"Jumlah fitur kategorikal : {len(cat_cols)}\n")

# =========================
# Missing Value Sebelum Imputasi
# =========================
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]

print(
    "Jumlah kolom dengan missing value sebelum imputasi:",
    len(missing_before),
    "\n"
)

# =========================
# Imputasi Numerik (Median)
# =========================
for col in num_cols:
    if df[col].isnull().any():
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

print("Imputasi fitur numerik dengan median selesai.")

# =========================
# Imputasi Kategorikal (Modus)
# =========================
for col in cat_cols:
    if df[col].isnull().any():
        # handling edge case: semua nilai NaN
        if df[col].dropna().empty:
            df[col] = df[col].fillna("Unknown")
        else:
            mode_value = df[col].mode(dropna=True)[0]
            df[col] = df[col].fillna(mode_value)

print("Imputasi fitur kategorikal dengan modus selesai.\n")

# =========================
# Validasi Akhir Missing Value
# =========================
remaining_missing = int(df.isnull().sum().sum())

if remaining_missing == 0:
    print("✔ Tidak ada missing value tersisa setelah imputasi.")
else:
    print(f"✖ Masih terdapat {remaining_missing:,} missing value setelah imputasi.")

print("\n=== IMPUTATION SELESAI ===")

print("=== ENCODING & SCALING ===\n")

from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
import pandas as pd

# =========================
# [1] Kolom Khusus (Tidak Diproses)
# =========================
ID_COLS = ["TransactionID"]
TIME_COLS = ["TransactionDT"]
SEQ_KEY_COLS = ["card1"]

SPECIAL_COLS = list(set(ID_COLS + TIME_COLS + SEQ_KEY_COLS))
SPECIAL_COLS = [c for c in SPECIAL_COLS if c in df.columns]

print(f"Kolom khusus (dikecualikan dari encoding & scaling): {SPECIAL_COLS}\n")

df_special = df[SPECIAL_COLS].copy()
df_proc = df.drop(columns=SPECIAL_COLS)

# =========================
# [2] Identifikasi Tipe Fitur
# =========================
num_cols = df_proc.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

cat_cols = df_proc.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(f"Jumlah fitur numerik     : {len(num_cols)}")
print(f"Jumlah fitur kategorikal : {len(cat_cols)}\n")

# =========================
# [3] Encoding Kategorikal
# =========================
if len(cat_cols) > 0:
    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    df_proc[cat_cols] = ordinal_encoder.fit_transform(df_proc[cat_cols])
    print("[1] Ordinal Encoding selesai.")
else:
    print("[1] Tidak ada fitur kategorikal untuk di-encode.")

# =========================
# [4] Scaling Numerik
# =========================
if len(num_cols) > 0:
    scaler = MinMaxScaler(feature_range=(0, 1))
    df_proc[num_cols] = scaler.fit_transform(df_proc[num_cols])
    print("[2] MinMax Scaling selesai.")
else:
    print("[2] Tidak ada fitur numerik untuk di-scale.")

# =========================
# [5] Gabungkan Kembali
# =========================
df_encoded_scaled = pd.concat(
    [
        df_special.reset_index(drop=True),
        df_proc.reset_index(drop=True)
    ],
    axis=1
)

print("\nShape akhir setelah encoding & scaling:", df_encoded_scaled.shape)
print("=== ENCODING & SCALING SELESAI ===")

=== IMPUTATION: NUMERIK & KATEGORIKAL ===

Jumlah fitur numerik     : 379
Jumlah fitur kategorikal : 21

Jumlah kolom dengan missing value sebelum imputasi: 0 

Imputasi fitur numerik dengan median selesai.
Imputasi fitur kategorikal dengan modus selesai.

✔ Tidak ada missing value tersisa setelah imputasi.

=== IMPUTATION SELESAI ===
=== ENCODING & SCALING ===

Kolom khusus (dikecualikan dari encoding & scaling): ['TransactionDT', 'TransactionID', 'card1']

Jumlah fitur numerik     : 376
Jumlah fitur kategorikal : 21

[1] Ordinal Encoding selesai.
[2] MinMax Scaling selesai.

Shape akhir setelah encoding & scaling: (144233, 402)
=== ENCODING & SCALING SELESAI ===


**SEQUENCE FORMULATION**

In [9]:
print("=== SEQUENCE FORMATION (LSTM) ===\n")

import numpy as np

# =========================
# [1] Parameter Sequence
# =========================
SEQ_LEN = 5
GROUP_COL = "card1"
TIME_COL = "TransactionDT"

# =========================
# [2] Validasi Kolom
# =========================
assert GROUP_COL in df_encoded_scaled.columns, f"{GROUP_COL} tidak ditemukan"
assert TIME_COL in df_encoded_scaled.columns, f"{TIME_COL} tidak ditemukan"

# =========================
# [3] Persiapan Data
# =========================
df_seq = df_encoded_scaled.copy()

# pastikan label align
assert len(df_seq) == len(y_merged), "Mismatch df dan label"

df_seq["_label"] = y_merged.values

# Urutkan berdasarkan group & waktu
df_seq = df_seq.sort_values(
    by=[GROUP_COL, TIME_COL],
    ascending=[True, True]
).reset_index(drop=True)

# =========================
# [4] Tentukan Fitur Sequence
# =========================
EXCLUDE_COLS = ["TransactionID", TIME_COL, GROUP_COL, "_label"]
feature_cols = [c for c in df_seq.columns if c not in EXCLUDE_COLS]

print(f"Jumlah fitur dalam sequence : {len(feature_cols)}")

# =========================
# [5] Build Sequence
# =========================
X_seq = []
y_seq = []

n_group_total = 0
n_group_used = 0

for card_id, group in df_seq.groupby(GROUP_COL, sort=False):
    n_group_total += 1

    group_len = len(group)
    if group_len < SEQ_LEN:
        continue

    n_group_used += 1

    group_values = group[feature_cols].to_numpy(dtype=np.float32)
    group_labels = group["_label"].to_numpy(dtype=np.int8)

    # sliding window
    for i in range(group_len - SEQ_LEN + 1):
        X_seq.append(group_values[i:i+SEQ_LEN])
        y_seq.append(group_labels[i + SEQ_LEN - 1])

# =========================
# [6] Convert ke NumPy
# =========================
X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.int8)

# =========================
# [7] Validasi Output
# =========================
print("\nSequence formation selesai.")
print(f"Total group card            : {n_group_total:,}")
print(f"Group digunakan (>=SEQ_LEN) : {n_group_used:,}")
print(f"X_seq shape                 : {X_seq.shape} (samples, timesteps, features)")
print(f"y_seq shape                 : {y_seq.shape}")

# =========================
# [8] Safety Check
# =========================
if len(X_seq) == 0:
    raise ValueError("Sequence kosong! Periksa SEQ_LEN atau distribusi data.")

print("\n=== SEQUENCE FORMATION SELESAI ===")

=== SEQUENCE FORMATION (LSTM) ===

Jumlah fitur dalam sequence : 399

Sequence formation selesai.
Total group card            : 8,499
Group digunakan (>=SEQ_LEN) : 2,629
X_seq shape                 : (123315, 5, 399) (samples, timesteps, features)
y_seq shape                 : (123315,)

=== SEQUENCE FORMATION SELESAI ===


**SPLIT DATA**

In [10]:
print("=== STRATIFIED SPLIT: TRAIN 64% | VAL 16% | TEST 20% ===\n")

from sklearn.model_selection import train_test_split
import numpy as np

# =========================
# [0] Validasi Awal
# =========================
assert len(X_seq) == len(y_seq), "Mismatch X_seq dan y_seq"

unique_classes = np.unique(y_seq)
assert len(unique_classes) == 2, "Label harus binary untuk stratified split"

# =========================
# [1] Parameter Proporsi
# =========================
TEST_SIZE = 0.20
VAL_SIZE_FROM_REMAIN = 0.20  # 16% dari total

# =========================
# [2] Split Test (20%)
# =========================
X_remain, X_test, y_remain, y_test = train_test_split(
    X_seq,
    y_seq,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_seq
)

print("✔ Split Test (20%) selesai.")

# =========================
# [3] Split Train & Validation
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X_remain,
    y_remain,
    test_size=VAL_SIZE_FROM_REMAIN,
    random_state=SEED,
    stratify=y_remain
)

print("✔ Split Train (64%) & Validation (16%) selesai.")

# =========================
# [4] Fungsi Ringkasan
# =========================
def dataset_summary(X, y, name):
    n_seq, seq_len, n_feat = X.shape

    n_fraud = int(y.sum())
    n_nonfraud = len(y) - n_fraud

    fraud_pct = (n_fraud / len(y)) * 100
    nonfraud_pct = (n_nonfraud / len(y)) * 100

    print(f"\n--- {name.upper()} SET ---")
    print(f"Jumlah sequence : {n_seq:,}")
    print(f"Sequence length : {seq_len}")
    print(f"Jumlah fitur    : {n_feat}")
    print(f"Non-Fraud (0)   : {n_nonfraud:,} ({nonfraud_pct:.4f}%)")
    print(f"Fraud (1)       : {n_fraud:,} ({fraud_pct:.4f}%)")

# =========================
# [5] Tampilkan Ringkasan
# =========================
dataset_summary(X_train, y_train, "Train")
dataset_summary(X_val,   y_val,   "Validation")
dataset_summary(X_test,  y_test,  "Test")

# =========================
# [6] Validasi Proporsi
# =========================
total_seq = len(X_train) + len(X_val) + len(X_test)

print("\n=== PROPORSI DATA (FINAL) ===")
print(f"Train      : {len(X_train) / total_seq:.2%}")
print(f"Validation : {len(X_val)   / total_seq:.2%}")
print(f"Test       : {len(X_test)  / total_seq:.2%}")

# =========================
# [7] Safety Check Imbalance
# =========================
def check_min_class(y, name):
    counts = np.bincount(y)
    if np.min(counts) < 5:
        print(f"⚠️ Warning: {name} sangat sedikit kelas minoritas:", counts)

check_min_class(y_train, "Train")
check_min_class(y_val, "Validation")
check_min_class(y_test, "Test")

=== STRATIFIED SPLIT: TRAIN 64% | VAL 16% | TEST 20% ===

✔ Split Test (20%) selesai.
✔ Split Train (64%) & Validation (16%) selesai.

--- TRAIN SET ---
Jumlah sequence : 78,921
Sequence length : 5
Jumlah fitur    : 399
Non-Fraud (0)   : 72,238 (91.5320%)
Fraud (1)       : 6,683 (8.4680%)

--- VALIDATION SET ---
Jumlah sequence : 19,731
Sequence length : 5
Jumlah fitur    : 399
Non-Fraud (0)   : 18,060 (91.5311%)
Fraud (1)       : 1,671 (8.4689%)

--- TEST SET ---
Jumlah sequence : 24,663
Sequence length : 5
Jumlah fitur    : 399
Non-Fraud (0)   : 22,575 (91.5339%)
Fraud (1)       : 2,088 (8.4661%)

=== PROPORSI DATA (FINAL) ===
Train      : 64.00%
Validation : 16.00%
Test       : 20.00%


In [11]:
print("=== SMOTE BALANCING FOR TRAIN ONLY ===\n")

# =====================================================
# IMPORT LIBRARY
# =====================================================
from imblearn.over_sampling import SMOTE
import numpy as np

# =====================================================
# [1] VALIDASI DATA
# =====================================================
print("=== VALIDASI DATA ===")

assert len(X_train) == len(y_train), (
    "Mismatch X_train dan y_train"
)

assert len(X_val) == len(y_val), (
    "Mismatch X_val dan y_val"
)

assert len(X_test) == len(y_test), (
    "Mismatch X_test dan y_test"
)

print("✔ Validasi dataset berhasil.\n")

# =====================================================
# [2] INFORMASI SEQUENCE
# =====================================================
seq_len = X_train.shape[1]
n_features = X_train.shape[2]

print("=== INFORMASI SEQUENCE ===")

print(f"Sequence Length : {seq_len}")
print(f"Jumlah Fitur    : {n_features}")

# =====================================================
# [3] DISTRIBUSI SEBELUM SMOTE
# =====================================================
print("\n=== DISTRIBUSI SEBELUM SMOTE ===")

print("\nTRAIN")
print(np.bincount(y_train))

print("\nVALIDATION")
print(np.bincount(y_val))

print("\nTEST")
print(np.bincount(y_test))

# =====================================================
# [4] FLATTEN TRAIN DATA
# =====================================================
# SMOTE hanya menerima input 2D
# sehingga sequence train perlu
# diubah sementara menjadi bentuk 2D
# =====================================================
print("\n=== FLATTEN TRAIN DATA ===")

X_train_flat = X_train.reshape(
    X_train.shape[0],
    -1
)

print(f"X_train_flat : {X_train_flat.shape}")

# =====================================================
# [5] SMOTE OBJECT
# =====================================================
# sampling_strategy = 0.5
#
# Fraud akan menjadi:
# 50% dari jumlah non-fraud
#
# Lebih stabil dibanding full 1:1
# =====================================================
smote = SMOTE(
    sampling_strategy=0.5,
    random_state=SEED,
    k_neighbors=5
)

# =====================================================
# [6] SMOTE TRAIN ONLY
# =====================================================
print("\n=== SMOTE TRAIN ===")

X_train_balanced_flat, y_train_balanced = smote.fit_resample(
    X_train_flat,
    y_train
)

print("✔ Train balancing selesai.")

# =====================================================
# [7] DISTRIBUSI SETELAH SMOTE
# =====================================================
print("\n=== DISTRIBUSI SETELAH SMOTE ===")

print("\nTRAIN")
print(np.bincount(y_train_balanced))

print("\nVALIDATION (ASLI)")
print(np.bincount(y_val))

print("\nTEST (ASLI)")
print(np.bincount(y_test))

# =====================================================
# [8] RESHAPE TRAIN KEMBALI KE 3D
# =====================================================
print("\n=== RESHAPE TRAIN KE SEQUENCE ===")

X_train_balanced = X_train_balanced_flat.reshape(
    -1,
    seq_len,
    n_features
)

print("✔ Reshape sequence selesai.")

# =====================================================
# [9] VALIDASI FINAL
# =====================================================
assert X_train_balanced.shape[1] == seq_len
assert X_train_balanced.shape[2] == n_features

print("\n✔ Validasi reshape berhasil.")

# =====================================================
# [10] INFORMASI SHAPE FINAL
# =====================================================
print("\n=== SHAPE FINAL ===")

print("\nTRAIN (SMOTE)")
print(f"X_train_balanced : {X_train_balanced.shape}")
print(f"y_train_balanced : {y_train_balanced.shape}")

print("\nVALIDATION (ASLI)")
print(f"X_val : {X_val.shape}")
print(f"y_val : {y_val.shape}")

print("\nTEST (ASLI)")
print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")

# =====================================================
# [11] RINGKASAN FINAL
# =====================================================
print("\n=== RINGKASAN BALANCING ===")

print("✔ Train menggunakan SMOTE")
print("✔ Validation menggunakan data asli")
print("✔ Test menggunakan data asli")

print("\n=== SMOTE BALANCING SELESAI ===")

=== SMOTE BALANCING FOR TRAIN ONLY ===

=== VALIDASI DATA ===
✔ Validasi dataset berhasil.

=== INFORMASI SEQUENCE ===
Sequence Length : 5
Jumlah Fitur    : 399

=== DISTRIBUSI SEBELUM SMOTE ===

TRAIN
[72238  6683]

VALIDATION
[18060  1671]

TEST
[22575  2088]

=== FLATTEN TRAIN DATA ===
X_train_flat : (78921, 1995)

=== SMOTE TRAIN ===
✔ Train balancing selesai.

=== DISTRIBUSI SETELAH SMOTE ===

TRAIN
[72238 36119]

VALIDATION (ASLI)
[18060  1671]

TEST (ASLI)
[22575  2088]

=== RESHAPE TRAIN KE SEQUENCE ===
✔ Reshape sequence selesai.

✔ Validasi reshape berhasil.

=== SHAPE FINAL ===

TRAIN (SMOTE)
X_train_balanced : (108357, 5, 399)
y_train_balanced : (108357,)

VALIDATION (ASLI)
X_val : (19731, 5, 399)
y_val : (19731,)

TEST (ASLI)
X_test : (24663, 5, 399)
y_test : (24663,)

=== RINGKASAN BALANCING ===
✔ Train menggunakan SMOTE
✔ Validation menggunakan data asli
✔ Test menggunakan data asli

=== SMOTE BALANCING SELESAI ===


**PELATIHAN MODEL**

**SDAE**

In [12]:
print("=== SDAE FOR FRAUD DETECTION ===")

# =====================================================
# IMPORT LIBRARY
# =====================================================
import os
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# =====================================================
# RANDOM SEED
# =====================================================
import random

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

tf.config.experimental.enable_op_determinism()

# =====================================================
# [1] SDAE MODEL
# =====================================================
def build_sdae(
    input_dim,
    latent_dim=64,
    noise_std=0.1
):

    inputs = layers.Input(
        shape=(input_dim,)
    )

    # ==========================
    # NOISE
    # ==========================
    x = layers.GaussianNoise(
        noise_std
    )(inputs)

    # ==========================
    # ENCODER
    # ==========================
    x = layers.Dense(
        256,
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.2)(x)

    x = layers.Dense(
        128,
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.2)(x)

    latent = layers.Dense(
        latent_dim,
        activation="relu",
        name="latent"
    )(x)

    # ==========================
    # DECODER
    # ==========================
    x = layers.Dense(
        128,
        activation="relu"
    )(latent)

    x = layers.Dense(
        256,
        activation="relu"
    )(x)

    outputs = layers.Dense(
        input_dim,
        activation="linear"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="Pure_SDAE"
    )

    return model

# =====================================================
# SDAE CLASSIFIER
# =====================================================
def build_sdae_classifier(
    encoder,
    input_dim
):

    inputs = layers.Input(
        shape=(input_dim,)
    )

    x = encoder(inputs)

    x = layers.Dense(
        64,
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="SDAE_Classifier"
    )

    return model

# =====================================================
# [2] THRESHOLD CONFIGURATION
# =====================================================
THRESHOLD = 0.3

# =====================================================
# [3] EXPERIMENT CONFIGURATION
# =====================================================
EXPERIMENT_NAME = "SMOTE_CW_1_7"

CLASS_WEIGHT = {
    0: 1,
    1: 7
}

# =====================================================
# RESULT DIRECTORY
# =====================================================
RESULT_DIR = "results"

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

# =====================================================
# [4] INITIALIZATION
# =====================================================
exp_dir = os.path.join(
    RESULT_DIR,
    EXPERIMENT_NAME
)

os.makedirs(
    exp_dir,
    exist_ok=True
)

print("\n" + "="*80)
print(f"EXPERIMENT : {EXPERIMENT_NAME}")
print("="*80)

# =================================================
# DATASET SELECTION
# =================================================
X_train_used = X_train_balanced
y_train_used = y_train_balanced

print("\nUsing SMOTE Train Data")

# =================================================
# FLATTEN DATA
# =================================================
X_train_tab = X_train_used.reshape(
    X_train_used.shape[0],
    -1
).astype(np.float32)

X_val_tab = X_val.reshape(
    X_val.shape[0],
    -1
).astype(np.float32)

X_test_tab = X_test.reshape(
    X_test.shape[0],
    -1
).astype(np.float32)

input_dim = X_train_tab.shape[1]

print("\n=== DATA SHAPE ===")

print("Train :", X_train_tab.shape)
print("Val   :", X_val_tab.shape)
print("Test  :", X_test_tab.shape)

# =================================================
# BUILD PURE SDAE
# =================================================
pure_sdae = build_sdae(
    input_dim=input_dim
)

# =================================================
# COMPILE PURE SDAE
# =================================================
pure_sdae.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="mse"
)


# =================================================
# TRAIN PURE SDAE
# =================================================
print("\n=== TRAINING PURE SDAE ===\n")

history_sdae = pure_sdae.fit(

    X_train_tab,
    X_train_tab,

    validation_data=(
        X_val_tab,
        X_val_tab
    ),

    epochs=50,

    batch_size=128,

    callbacks=[

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1
        )

    ],

    verbose=1
)

pd.DataFrame(
    history_sdae.history
).to_csv(

    os.path.join(
        exp_dir,
        "sdae_history.csv"
    ),

    index=False
)

# =================================================
# EXTRACT ENCODER
# =================================================
encoder = models.Model(

    inputs=pure_sdae.input,

    outputs=pure_sdae.get_layer(
        "latent"
    ).output
)

encoder.trainable = True

# =================================================
# BUILD CLASSIFIER
# =================================================
sdae = build_sdae_classifier(

    encoder=encoder,

    input_dim=input_dim
)

# =================================================
# COMPILE CLASSIFIER
# =================================================
sdae.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

# =================================================
# TRAIN CLASSIFIER
# =================================================
print("\n=== TRAINING SDAE CLASSIFIER ===\n")

history_classifier = sdae.fit(

    X_train_tab,
    y_train_used,

    validation_data=(
        X_val_tab,
        y_val
    ),

    epochs=50,

    batch_size=128,

    class_weight=CLASS_WEIGHT,

    callbacks=[

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1
        )

    ],

    verbose=1
)

model_path = os.path.join(
    exp_dir,
    "model.keras"
)

sdae.save(model_path)

print(
    f"Model saved : {model_path}"
)

pd.DataFrame(
    history_classifier.history
).to_csv(

    os.path.join(
        exp_dir,
        "classifier_history.csv"
    ),

    index=False
)


# =================================================
# PREDICT PROBABILITY
# =================================================
print("\n=== PREDICT TEST SET ===")

y_test_prob = sdae.predict(
    X_test_tab,
    verbose=0
).ravel()

np.save(
    os.path.join(
        exp_dir,
        "y_test_prob.npy"
    ),
    y_test_prob
)

# =================================================
# AUC ROC
# =================================================
auc_score = roc_auc_score(
    y_test,
    y_test_prob
)

print(f"\nAUC-ROC : {auc_score:.4f}")

# =====================================================
# CLASSIFICATION THRESHOLD
# =====================================================
print("\n" + "-"*70)
print(f"THRESHOLD : {THRESHOLD}")
print("-"*70)

y_test_pred = (
    y_test_prob >= THRESHOLD
).astype(int)

# =============================================
# METRICS
# =============================================
acc = accuracy_score(
    y_test,
    y_test_pred
)

prec = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

rec = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

f1_macro = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

cm = confusion_matrix(
    y_test,
    y_test_pred
)

tn, fp, fn, tp = cm.ravel()

# =============================================
# PRINT METRICS
# =============================================
print("\n=== EVALUATION RESULTS ===")

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"F1-Macro  : {f1_macro:.4f}")
print(f"AUC-ROC   : {auc_score:.4f}")

# =============================================
# CONFUSION MATRIX
# =============================================
print("\n=== CONFUSION MATRIX ===")

print(cm)

print(f"\nTN : {tn}")
print(f"FP : {fp}")
print(f"FN : {fn}")
print(f"TP : {tp}")

# =============================================
# CLASSIFICATION REPORT
# =============================================
print("\n=== CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_test,
        y_test_pred,
        digits=4
    )
)


print("\n=== SDAE FINISHED ===")

=== SDAE FOR FRAUD DETECTION ===

EXPERIMENT : SMOTE_CW_1_7

Using SMOTE Train Data

=== DATA SHAPE ===
Train : (108357, 1995)
Val   : (19731, 1995)
Test  : (24663, 1995)

=== TRAINING PURE SDAE ===

Epoch 1/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 1955.6179 - val_loss: 309.7945 - learning_rate: 1.0000e-04
Epoch 2/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 328.9998 - val_loss: 285.1720 - learning_rate: 1.0000e-04
Epoch 3/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 309.1108 - val_loss: 274.7393 - learning_rate: 1.0000e-04
Epoch 4/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 292.6380 - val_loss: 258.7898 - learning_rate: 1.0000e-04
Epoch 5/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 272.8509 - val_loss: 236.7711 - learning_rate: 1.0000e-04
Epoch 6/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 244.0738 - val_loss: 205.5580 - learning_rate: 1.0000e-04
Epoch 7/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 203.8653 - val_loss: 160.9833 

**LSTM-SDAE**

In [13]:
print("=== LSTM-SDAE FOR FRAUD DETECTION ===")

# =====================================================
# IMPORT LIBRARY
# =====================================================

import os
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# =====================================================
# RANDOM SEED
# =====================================================
import random

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

tf.config.experimental.enable_op_determinism()

# =====================================================
# [2] BUILD SDAE
# =====================================================
def build_sdae(
    input_dim,
    latent_dim=64,
    noise_std=0.1
):

    inputs = layers.Input(
        shape=(input_dim,)
    )

    # =================================================
    # GAUSSIAN NOISE
    # =================================================
    x = layers.GaussianNoise(
        noise_std
    )(inputs)

    # =================================================
    # ENCODER BLOCK 1
    # =================================================
    x = layers.Dense(
        256,
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.2)(x)

    # =================================================
    # ENCODER BLOCK 2
    # =================================================
    x = layers.Dense(
        128,
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.2)(x)

    # =================================================
    # LATENT REPRESENTATION
    # =================================================
    latent = layers.Dense(
        latent_dim,
        activation="relu",
        name="latent"
    )(x)

    # =================================================
    # DECODER
    # =================================================
    x = layers.Dense(
        128,
        activation="relu"
    )(latent)

    x = layers.Dense(
        256,
        activation="relu"
    )(x)

    outputs = layers.Dense(
        input_dim,
        activation="linear"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="SDAE"
    )

    return model

# =====================================================
# [3] BUILD LSTM-SDAE
# =====================================================
def build_lstm_sdae(
    seq_len,
    feature_dim,
    encoder_model
):

    inputs = layers.Input(
        shape=(seq_len, feature_dim)
    )

    # =================================================
    # TIMEDISTRIBUTED ENCODER
    # =================================================
    x = layers.TimeDistributed(
        encoder_model
    )(inputs)

    # =================================================
    # BIDIRECTIONAL LSTM 1
    # =================================================
    x = layers.Bidirectional(

        layers.LSTM(
            128,
            return_sequences=True,
            dropout=0.2
        )

    )(x)

    # =================================================
    # BIDIRECTIONAL LSTM 2
    # =================================================
    x = layers.Bidirectional(

        layers.LSTM(
            64,
            return_sequences=False,
            dropout=0.2
        )

    )(x)

    # =================================================
    # DENSE CLASSIFIER
    # =================================================
    x = layers.Dense(
        128,
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.3)(x)

    x = layers.Dense(
        64,
        activation="relu"
    )(x)

    x = layers.Dropout(0.2)(x)

    # =================================================
    # OUTPUT
    # =================================================
    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="Final_LSTM_SDAE"
    )

    return model

# =====================================================
# [4] THRESHOLD LIST
# =====================================================
THRESHOLD = 0.3

# =====================================================
# [5] EXPERIMENT CONFIGURATION
# =====================================================
EXPERIMENT_NAME = "SMOTE_CW_1_7"

CLASS_WEIGHT = {
    0: 1,
    1: 7
}

RESULT_DIR = "results_lstm_sdae"

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

# =====================================================
# [6] START EXPERIMENT
# =====================================================
exp_dir = os.path.join(
    RESULT_DIR,
    EXPERIMENT_NAME
)

os.makedirs(
    exp_dir,
    exist_ok=True
)

print("\n" + "="*80)
print(f"EXPERIMENT : {EXPERIMENT_NAME}")
print("="*80)


# =================================================
# DATASET SELECTION
# =================================================

X_train_used = X_train_balanced
y_train_used = y_train_balanced

print("\nUsing SMOTE Train Data")

# =================================================
# PREPARE SDAE DATA
# =================================================
X_train_step = X_train_used.reshape(
    -1,
    X_train_used.shape[2]
).astype(np.float32)

X_val_step = X_val.reshape(
    -1,
    X_val.shape[2]
).astype(np.float32)

feature_dim = X_train_used.shape[2]
seq_len = X_train_used.shape[1]

print("\n=== DATA SHAPE ===")

print("Train Step :", X_train_step.shape)
print("Val Step   :", X_val_step.shape)

# =================================================
# BUILD SDAE
# =================================================
sdae = build_sdae(
    input_dim=feature_dim
)

# =================================================
# COMPILE SDAE
# =================================================
sdae.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="mse"
)

# =================================================
# TRAIN SDAE
# =================================================
print("\n=== TRAINING SDAE ===\n")

history_sdae = sdae.fit(

    X_train_step,
    X_train_step,

    validation_data=(
        X_val_step,
        X_val_step
    ),

    epochs=50,

    batch_size=512,

    callbacks=[

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1
        )

    ],

    verbose=1
)

pd.DataFrame(
    history_sdae.history
).to_csv(
    os.path.join(
        exp_dir,
        "sdae_history.csv"
    ),
    index=False
)

# =================================================
# EXTRACT ENCODER
# =================================================
encoder = models.Model(
    inputs=sdae.input,
    outputs=sdae.get_layer(
        "latent"
    ).output
)

encoder.trainable = True

# =================================================
# BUILD LSTM-SDAE
# =================================================
lstm_sdae = build_lstm_sdae(
    seq_len=seq_len,
    feature_dim=feature_dim,
    encoder_model=encoder
)

print("\n=== MODEL SUMMARY ===")

lstm_sdae.summary()

# =================================================
# COMPILE MODEL
# =================================================
lstm_sdae.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
    ),

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

# =================================================
# TRAINING MODEL
# =================================================
print("\n=== TRAINING LSTM-SDAE ===\n")

history_lstm = lstm_sdae.fit(

    X_train_used,
    y_train_used,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=50,

    batch_size=128,

    class_weight=CLASS_WEIGHT,

    callbacks=[

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1
        )

    ],

    verbose=1
)

pd.DataFrame(
    history_lstm.history
).to_csv(
    os.path.join(
        exp_dir,
        "lstm_history.csv"
    ),
    index=False
)

model_path = os.path.join(
exp_dir,
"model.keras"
)

lstm_sdae.save(model_path)

print(
    f"Model saved : {model_path}"
)

# =================================================
# PREDICT PROBABILITY
# =================================================
print("\n=== PREDICT TEST SET ===")

y_test_prob = lstm_sdae.predict(
    X_test,
    verbose=0
).ravel()

np.save(

    os.path.join(
        exp_dir,
        "y_test_prob.npy"
    ),

    y_test_prob
)

# =================================================
# AUC ROC
# =================================================
auc_score = roc_auc_score(
    y_test,
    y_test_prob
)

print(f"\nAUC-ROC : {auc_score:.4f}")

# =================================================
# THRESHOLD ANALYSIS
# =================================================
print("\n" + "-"*70)
print(f"THRESHOLD : {THRESHOLD}")
print("-"*70)

y_test_pred = (
    y_test_prob >= THRESHOLD
).astype(int)

# =============================================
# METRICS
# =============================================
acc = accuracy_score(
    y_test,
    y_test_pred
)

prec = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

rec = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

f1_macro = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

# =============================================
# PRINT METRICS
# =============================================
print("\n=== EVALUATION RESULTS ===")

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"F1-Macro  : {f1_macro:.4f}")
print(f"AUC-ROC   : {auc_score:.4f}")

# =============================================
# CONFUSION MATRIX
# =============================================
print("\n=== CONFUSION MATRIX ===")

cm = confusion_matrix(
    y_test,
    y_test_pred
)

print(cm)

tn, fp, fn, tp = cm.ravel()



print(f"\nTN : {tn}")
print(f"FP : {fp}")
print(f"FN : {fn}")
print(f"TP : {tp}")

# =============================================
# CLASSIFICATION REPORT
# =============================================
print("\n=== CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_test,
        y_test_pred,
        digits=4
    )
)



print("\n=== LSTM-SDAE FINISHED ===")

=== LSTM-SDAE FOR FRAUD DETECTION ===

EXPERIMENT : SMOTE_CW_1_7

Using SMOTE Train Data

=== DATA SHAPE ===
Train Step : (541785, 399)
Val Step   : (98655, 399)

=== TRAINING SDAE ===

Epoch 1/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - loss: 1337.4377 - val_loss: 20.7154 - learning_rate: 1.0000e-04
Epoch 2/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 52.6939 - val_loss: 10.6073 - learning_rate: 1.0000e-04
Epoch 3/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 36.7070 - val_loss: 6.3789 - learning_rate: 1.0000e-04
Epoch 4/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 29.1285 - val_loss: 5.2852 - learning_rate: 1.0000e-04
Epoch 5/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 25.1960 - val_loss: 4.4194 - learning_rate: 1.0000e-04
Epoch 6/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 22.9167 - val_loss: 3.9380 - learning_rate: 1.0000e-04
Epoch 7/50
1059/1059 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 21.4395 - val_loss: 3.7421 - learning_rate: 1

Model: "Final_LSTM_SDAE"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 5, 399)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 5, 64)          │       145,088 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 5, 256)         │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 532,417 (2.03 MB)

 Trainable params: 531,393 (2.03 MB)

 Non-trainable params: 1,024 (4.00 KB)


=== TRAINING LSTM-SDAE ===

Epoch 1/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 26s 21ms/step - accuracy: 0.4028 - loss: 1.7327 - val_accuracy: 0.2051 - val_loss: 1.4407 - learning_rate: 1.0000e-04
Epoch 2/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.3944 - loss: 1.5850 - val_accuracy: 0.2330 - val_loss: 1.3396 - learning_rate: 1.0000e-04
Epoch 3/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.4022 - loss: 1.4904 - val_accuracy: 0.5694 - val_loss: 0.7409 - learning_rate: 1.0000e-04
Epoch 4/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.5119 - loss: 1.3090 - val_accuracy: 0.3068 - val_loss: 1.4708 - learning_rate: 1.0000e-04
Epoch 5/50
847/847 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5837 - loss: 1.2145
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
847/847 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.5954 - loss: 1.1870 - val_accuracy: 0.3624 - val_loss: 1.2087 - learning_rate: 1.0000e-04
Epoch 6/50
847/847 ━━━━━━━━━━